In [ ]:
!pip install -qU  unsloth
!pip install -qU  trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.9/511.9 kB 12.0 MB/s eta 0:00:00


In [ ]:
# !pip uninstall -y trl

Found existing installation: trl 0.21.0
Uninstalling trl-0.21.0:
  Successfully uninstalled trl-0.21.0


In [ ]:
# !pip cache purge

Files removed: 192


### Generating synthetic data

In [ ]:
import pandas as pd
from unsloth import FastLanguageModel
from trl import SFTTrainer, GRPOConfig, GRPOTrainer
from transformers import TrainingArguments
import torch
from datasets import Dataset
import json
import random
import trl
print(f"TRL Version after install: {trl.__version__}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
TRL Version after install: 0.21.0


In [ ]:
df = pd.read_json('/content/o6ai-agent-hr/synthetic_resume_eval_data_1000.json')
df.columns

Index(['id', 'source', 'specversion', 'type', 'time', 'subject',
       'datacontenttype', 'status', 'intent', 'response', 'similarity_score',
       'evaluation_json', 'candidate_name', 'candidate_phone', 'notes',
       'correlationid', 'traceid', 'userid', 'sessionid',
       'processing_duration_ms', 'model_version', 'confidence_score',
       'data_version', 'job_description_skills'],
      dtype='object')

In [ ]:
df.head()

In [ ]:
#Filter the dataset to keep only the columns needed for SFT
required_col = ['status' ,'evaluation_json','candidate_name' ,'candidate_phone', 'confidence_score', 'job_description_skills' ]
sft_dataset = df.loc[:, required_col]
sft_dataset.head()

In [ ]:
sft_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   status                  1000 non-null   object 
 1   evaluation_json         1000 non-null   object 
 2   candidate_name          1000 non-null   object 
 3   candidate_phone         1000 non-null   int64  
 4   confidence_score        1000 non-null   float64
 5   job_description_skills  1000 non-null   object 
dtypes: float64(1), int64(1), object(4)
memory usage: 47.0+ KB


In [ ]:
sft_dataset_train = sft_dataset[:900]
sft_dataset_eval = sft_dataset[900:1000]
print(len(sft_dataset_train) , len(sft_dataset_eval))

900 100


In [ ]:
#Model Configuration
max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2-0.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Set a padding token
tokenizer.pad_token = tokenizer.eos_token

#LoRA Configuration
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = True,
    random_state = 3407,
)

==((====))==  Unsloth 2025.7.7: Fast Qwen2 patching. Transformers: 4.53.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.31.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
hf_sft_dataset_train = Dataset.from_pandas(sft_dataset_train)
hf_sft_dataset_eval = Dataset.from_pandas(sft_dataset_eval)

In [ ]:
def format_for_sft(example):
    # Assuming 'evaluation_json' contains 'present_skills' as the candidate's skills
    # And 'job_description_skills' is the job description's skills

    # Safely parse evaluation_json
    try:
        evaluation_data = json.loads(example['evaluation_json'])
    except json.JSONDecodeError:
        print(f"Warning: Could not parse evaluation_json for example: {example.get('id', 'N/A')}")
        # Return a structure that won't cause issues, or skip this example if necessary
        return {"text": ""} # Or raise an error if you want to strictly filter malformed data

    candidate_skills = evaluation_data.get('skills_match', {}).get('present_skills', [])
    candidate_skills_str = ", ".join(candidate_skills) # Convert list to string for prompt

    # Extract candidate name and phone, ensuring they exist
    candidate_name = example.get('candidate_name', 'N/A')
    candidate_phone = example.get('candidate_phone', 'N/A')

    # Construct the prompt with candidate details
    prompt = (
        f"Please evaluate the following candidate's skills against the job description's required skills.\n\n"
        f"### Candidate Information:\n"
        f"Name: {candidate_name}\n"
        f"Phone: {candidate_phone}\n\n"
        f"### Candidate Skills:\n{candidate_skills_str}\n\n"
        f"### Job Description Skills:\n{example['job_description_skills']}"
    )

    # The response is still the full evaluation_json
    response = example['evaluation_json']

    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": response},
    ]

    # Ensure tokenizer is available in the scope where this function is called
    # Assuming tokenizer.apply_chat_template is how you format for your specific model
    return { "text": tokenizer.apply_chat_template(messages, tokenize=False) }

# Apply the formatting
formatted_sft_dataset_train = hf_sft_dataset_train.map(format_for_sft, remove_columns=required_col)
formatted_sft_dataset_eval = hf_sft_dataset_eval.map(format_for_sft, remove_columns=required_col)

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
from transformers import TrainingArguments

#SFT Training Configuration
training_args = TrainingArguments(
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 60,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs",
    report_to = "none",
    eval_strategy="steps",
    eval_steps=10,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_sft_dataset_train,
    eval_dataset=formatted_sft_dataset_eval,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = training_args,
)

# Start the training
trainer.train()

Unsloth: Tokenizing ["text"]:   0%|          | 0/900 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"]:   0%|          | 0/100 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 900 | Num Epochs = 2 | Total steps = 60
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Step,Training Loss,Validation Loss
10,0.893000,0.815490
20,0.543900,0.539958
30,0.523400,0.488675
40,0.446600,0.452533
50,0.436200,0.441303
60,0.419800,0.436404


TrainOutput(global_step=60, training_loss=0.7122296581665675, metrics={'train_runtime': 112.2064, 'train_samples_per_second': 8.556, 'train_steps_per_second': 0.535, 'total_flos': 428189826078720.0, 'train_loss': 0.7122296581665675})

In [ ]:
output_model_dir = "sft_qwen_resume_eval_model"

print(f"Saving the SFT-tuned model to {output_model_dir}...")
trainer.save_model(output_model_dir)

tokenizer.save_pretrained(output_model_dir)

print(f"Model and tokenizer saved successfully to {output_model_dir}.")

Saving the SFT-tuned model to sft_qwen_resume_eval_model...
Model and tokenizer saved successfully to sft_qwen_resume_eval_model.


## Load Model

In [ ]:
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from peft import PeftModel, prepare_model_for_kbit_training
import torch

# Paths
base_model_name = "unsloth/Qwen2-0.5B-Instruct-bnb-4bit"
lora_path = "/content/o6ai-agent-hr/sft_qwen_resume_eval_model"
max_seq_length = 2048

# Step 1: Load the base model using Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

tokenizer.pad_token = tokenizer.eos_token

FastLanguageModel.for_training(model)

model = PeftModel.from_pretrained(
    model,
    lora_path,
    torch_dtype=torch.float16,  # Match the base model dtype
    is_trainable=True  # Important for seeing trainable parameters
)

# Step 4: Confirm it worked
print("Trainable parameters:", model.print_trainable_parameters())

# Optional: Verify adapter is loaded
print("Loaded adapters:", model.peft_config.keys())

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.8.4: Fast Qwen2 patching. Transformers: 4.55.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.31.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


<string>:37: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.


model.safetensors:   0%|          | 0.00/457M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/80.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497
Trainable parameters: None
Loaded adapters: dict_keys(['default'])


## Generate result from sft model

In [ ]:
def create_evaluation_prompt(job_skills, candidate_profile):
    candidate_skills = candidate_profile["skills"]
    candidate_total_experience = candidate_profile["total_experience"]
    candidate_relevenet_experience = candidate_profile["relevant_experience"]
    candidate_previous_roles = candidate_profile["previous_roles"]
    candidate_education = candidate_profile["education"]
    candidate_certification = candidate_profile["certifications"]
    candidate_domain_expeience = candidate_profile["domain_experience"]

    prompt = f"""<|im_start|>system
            You are an HR expert evaluating candidate resumes. Provide a score (0-100), explanation, and status (SELECTED/REJECTED).
            <|im_end|>
            <|im_start|>user
            Job Requirements: {', '.join(job_skills)}

            Candidate Profile:
            - Skills: {', '.join(candidate_profile["skills"])}
            - Total Experience: {candidate_profile["total_experience"]}
            - Relevant Experience: {candidate_profile["relevant_experience"]}
            - Previous Roles: {', '.join(candidate_profile["previous_roles"])}
            - Education: {candidate_profile["education"]}
            - Certifications: {', '.join(candidate_profile["certifications"])}
            - Domain Experience: {', '.join(candidate_profile["domain_experience"])}

            Evaluate this candidate and provide:
            1. Score: (0-100) — based on how well the candidate's profile matches the job requirements.
            2. Explanation: For each job requirement, explain whether the candidate meets it and how (based on skills, experience, domain knowledge, etc.).
            3. Status: SELECTED or REJECTED (based on overall fit).

            <|im_end|>
            <|im_start|>assistant"""

    return prompt


def generate_qwen_response(job_skills, candidate_profile):
    prompt = create_evaluation_prompt(job_skills, candidate_profile)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=3000,
        do_sample=True,
        temperature=0.7,
        top_p = 0.9,
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the assistant's response part
    assistant_prefix = "<|im_start|>assistant"
    if assistant_prefix in generated_text:
        generated_text = generated_text.split(assistant_prefix, 1)[1].strip()
    return generated_text

In [ ]:
job_skills = ["Python", "TensorFlow", "PyTorch", "scikit-learn", "AWS", "GCP", "Docker", "Kubernetes", "MLflow", "Kubeflow"]
# candidate_skills = ["Python", "scikit-learn", "Pandas", "NumPy", "PyTorch", "AWS SageMaker", "Docker", "Git", "NLP", "Image Classification"]
candidate_profile = {
    "skills": ["Python", "scikit-learn", "Pandas", "NumPy", "PyTorch", "AWS SageMaker",
               "Docker", "Git", "NLP", "Image Classification", "REST APIs"],
    "total_experience": "4 years",
    "relevant_experience": "3 years in ML/AI",
    "previous_roles": ["ML Engineer at Tech Startup", "Data Scientist at Finance Corp"],
    "education": "M.S. in Computer Science",
    "certifications": ["AWS Machine Learning Specialty", "Google Cloud Professional ML Engineer"],
    "domain_experience": ["NLP", "Computer Vision", "Recommendation Systems"]
}


In [ ]:
prompt = create_evaluation_prompt(job_skills, candidate_profile)
print(prompt)

<|im_start|>system
            You are an HR expert evaluating candidate resumes. Provide a score (0-100), explanation, and status (SELECTED/REJECTED).
            <|im_end|>
            <|im_start|>user
            Job Requirements: Python, TensorFlow, PyTorch, scikit-learn, AWS, GCP, Docker, Kubernetes, MLflow, Kubeflow

            Candidate Profile:
            - Skills: Python, scikit-learn, Pandas, NumPy, PyTorch, AWS SageMaker, Docker, Git, NLP, Image Classification, REST APIs
            - Total Experience: 4 years
            - Relevant Experience: 3 years in ML/AI
            - Previous Roles: ML Engineer at Tech Startup, Data Scientist at Finance Corp
            - Education: M.S. in Computer Science
            - Certifications: AWS Machine Learning Specialty, Google Cloud Professional ML Engineer
            - Domain Experience: NLP, Computer Vision, Recommendation Systems

            Evaluate this candidate and provide:
            1. Score: (0-100) — based on how well t

In [ ]:
generated_response = generate_qwen_response(job_skills, candidate_profile)
print(generated_response)

system
            You are an HR expert evaluating candidate resumes. Provide a score (0-100), explanation, and status (SELECTED/REJECTED).
            
            user
            Job Requirements: Python, TensorFlow, PyTorch, scikit-learn, AWS, GCP, Docker, Kubernetes, MLflow, Kubeflow

            Candidate Profile:
            - Skills: Python, scikit-learn, Pandas, NumPy, PyTorch, AWS SageMaker, Docker, Git, NLP, Image Classification, REST APIs
            - Total Experience: 4 years
            - Relevant Experience: 3 years in ML/AI
            - Previous Roles: ML Engineer at Tech Startup, Data Scientist at Finance Corp
            - Education: M.S. in Computer Science
            - Certifications: AWS Machine Learning Specialty, Google Cloud Professional ML Engineer
            - Domain Experience: NLP, Computer Vision, Recommendation Systems

            Evaluate this candidate and provide:
            1. Score: (0-100) — based on how well the candidate's profile matches the

In [ ]:
job_skills_str = ", ".join(job_skills)

# Instead of candidate_skills_str, we now use the structured candidate_profile
# candidate_skills_str = ", ".join(candidate_skills)  # Remove this line

# Clean up the generated response to extract only the actual evaluation
clean_response = generated_response.strip()
if clean_response.startswith('{"score"'):
    # It's already clean JSON
    actual_response = clean_response
else:
    # Extract the JSON part if there's extra text
    import re
    json_match = re.search(r'\{"score".*?\}', clean_response)
    actual_response = json_match.group(0) if json_match else clean_response

# Updated to use candidate_profile structure
sft_output = [
    {
        'job_requirements': job_skills_str,
        'candidate_profile': candidate_profile,  # Use the structured profile
        'generated_response': actual_response,
        'label': 1  # 1 for good, 0 for bad (if you have ground truth)
    }
]

print("SFT Output:")
print(sft_output)

SFT Output:
[{'job_requirements': 'Python, TensorFlow, PyTorch, scikit-learn, AWS, GCP, Docker, Kubernetes, MLflow, Kubeflow', 'candidate_profile': {'skills': ['Python', 'scikit-learn', 'Pandas', 'NumPy', 'PyTorch', 'AWS SageMaker', 'Docker', 'Git', 'NLP', 'Image Classification', 'REST APIs'], 'total_experience': '4 years', 'relevant_experience': '3 years in ML/AI', 'previous_roles': ['ML Engineer at Tech Startup', 'Data Scientist at Finance Corp'], 'education': 'M.S. in Computer Science', 'certifications': ['AWS Machine Learning Specialty', 'Google Cloud Professional ML Engineer'], 'domain_experience': ['NLP', 'Computer Vision', 'Recommendation Systems']}, 'generated_response': '{"score": 85, "explanation": "Highly experienced, strong fit for senior roles.", "status": "SELECTED"}', 'label': 1}]


# Custom Reward Function


# Six Evaluation Parameters

1.   Status Correctness (20%): Checks SELECTED/REJECTED decision validity
2.   Skills Matching (20%): Analyzes alignment between candidate skills and job
requirements

3.   Experience Evaluation (15%): Assesses if score reflects experience level appropriately
4.   Education Assessment (10%): Considers education level in scoring

5.   Explanation Quality (10%): Evaluates completeness and reasoning in explanations

6.   Score Accuracy (25%): Evaluates if the model's 0-100 score is reasonable















In [3]:
"""
GRPO-Compatible Resume Evaluation Reward Function

This module implements a comprehensive reward function for GRPO training that evaluates
resume-job description matching using six key parameters:
1. Status Correctness (20%)
2. Skills Matching (20%)
3. Experience Evaluation (15%)
4. Education Assessment (10%)
5. Explanation Quality (10%)
6. Score Accuracy (25%)

Compatible with TRL GRPO Trainer requirements.
"""

import re
import json
import numpy as np
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass


@dataclass
class EvaluationScores:
    """Data class to hold all evaluation scores and metadata"""
    # Core scores (Si ∈ {-1, 0, 1})
    status_correctness: int = 0
    skills_matching: int = 0
    experience_evaluation: int = 0
    education_assessment: int = 0
    explanation_quality: int = 0
    score_accuracy: int = 0

    # Dynamic weight adjustments
    skills_weight_delta: float = 0.0
    experience_weight_delta: float = 0.0
    education_weight_delta: float = 0.0

    # Extracted data
    model_score: Optional[int] = None
    model_status: Optional[str] = None
    model_explanation: Optional[str] = None


class GRPOResumeRewardFunction:
    """
    GRPO-compatible reward function for HR resume evaluation.
    Implements: Reward = Σ(Wi * Si) where Wi ∈ [0,1] and Si ∈ {-1, 0, 1}
    """

    def __init__(self, weights: Dict[str, float] = None):
        """Initialize with parameter weights"""
        self.default_weights = {
            'score_accuracy': 0.25,        # How accurate is the model's score
            'status_correctness': 0.20,     # Correctness of SELECTED/REJECTED decision
            'skills_matching': 0.20,        # Quality of skills analysis
            'experience_evaluation': 0.15,  # Proper experience level classification
            'education_assessment': 0.10,   # Education level evaluation
            'explanation_quality': 0.10     # Quality and completeness of explanation
        }

        self.weights = weights if weights is not None else self.default_weights

        # Normalize weights to sum to 1.0
        total_weight = sum(self.weights.values())
        if abs(total_weight - 1.0) > 1e-6:
            self.weights = {k: v/total_weight for k, v in self.weights.items()}

    def extract_model_output(self, completion: str) -> Dict[str, Any]:
        """Extract structured data from model's completion"""
        try:
            # Try to parse JSON directly first
            json_match = re.search(r'\{[^{}]*"score"[^{}]*\}', completion, re.DOTALL)
            if not json_match:
                json_match = re.search(r'\{.*?\}', completion, re.DOTALL)

            if json_match:
                try:
                    data = json.loads(json_match.group(0))
                    score = data.get("score") or data.get("overall_score")
                    status = data.get("status", "").upper()
                    explanation = data.get("explanation", "")

                    return {
                        'model_score': int(score) if score is not None else None,
                        'model_status': status if status in ['SELECTED', 'REJECTED'] else None,
                        'model_explanation': explanation,
                        'raw_json': data
                    }
                except (json.JSONDecodeError, ValueError, TypeError):
                    pass

            # Fallback: extract using regex patterns
            score_match = re.search(r'["\']?score["\']?\s*:?\s*(\d+)', completion, re.IGNORECASE)
            status_match = re.search(r'["\']?status["\']?\s*:?\s*["\']?(SELECTED|REJECTED)["\']?', completion, re.IGNORECASE)
            explanation_match = re.search(r'["\']?explanation["\']?\s*:?\s*["\']?([^"\'}\n]+)', completion, re.IGNORECASE)

            return {
                'model_score': int(score_match.group(1)) if score_match else None,
                'model_status': status_match.group(1).upper() if status_match else None,
                'model_explanation': explanation_match.group(1) if explanation_match else "",
                'raw_json': None
            }

        except Exception as e:
            print(f"Warning: Error extracting model output: {e}")
            return {
                'model_score': None,
                'model_status': None,
                'model_explanation': "",
                'raw_json': None
            }

    def extract_skills_from_data(self, job_skills_str: str, evaluation_json_str: str) -> Tuple[List[str], List[str]]:
        """Extract job skills and candidate skills from dataset"""
        # Extract job skills
        job_skills = []
        if job_skills_str:
            job_skills = [skill.strip() for skill in job_skills_str.split(',')]

        # Extract candidate skills from evaluation JSON
        candidate_skills = []
        if evaluation_json_str:
            try:
                eval_data = json.loads(evaluation_json_str)
                skills_data = eval_data.get('skills_match', {})
                present_skills = skills_data.get('present_skills', [])
                candidate_skills = present_skills if isinstance(present_skills, list) else []
            except (json.JSONDecodeError, TypeError):
                pass

        return job_skills, candidate_skills

    def evaluate_score_accuracy(self, model_score: Optional[int], ground_truth_score: Optional[int]) -> int:
        """Evaluate accuracy of model's score (Si ∈ {-1, 0, 1})"""
        if model_score is None:
            return -1  # No score provided

        if not (0 <= model_score <= 100):
            return -1  # Invalid score range

        if ground_truth_score is not None:
            error = abs(model_score - ground_truth_score)
            if error <= 10:
                return 1   # Within 10 points
            elif error <= 20:
                return 0   # Within 20 points
            else:
                return -1  # More than 20 points off

        # Without ground truth, check if score is reasonable (not extreme)
        if 20 <= model_score <= 80:
            return 0   # Reasonable range
        elif model_score < 10 or model_score > 95:
            return -1  # Extreme scores need justification
        else:
            return 0   # Somewhat extreme but acceptable

    def evaluate_status_correctness(self, model_status: Optional[str], expected_status: Optional[str],
                                  model_score: Optional[int]) -> int:
        """Evaluate correctness of SELECTED/REJECTED decision"""
        if model_status not in ['SELECTED', 'REJECTED']:
            return -1  # Invalid or missing status

        if expected_status is not None:
            return 1 if model_status == expected_status else -1

        # If no expected status, check logical consistency with score
        if model_score is not None:
            if model_status == 'SELECTED' and model_score < 30:
                return -1  # Illogical: selecting very low score
            elif model_status == 'REJECTED' and model_score > 85:
                return -1  # Illogical: rejecting very high score
            else:
                return 0   # Consistent with score

        return 0  # Cannot evaluate without more context

    def evaluate_skills_matching(self, job_skills: List[str], candidate_skills: List[str],
                                model_score: Optional[int]) -> Tuple[int, float]:
        """Evaluate skills matching with dynamic weight adjustment"""
        if not job_skills or not candidate_skills:
            return -1, 0.0

        # Calculate skill match percentage
        job_skills_lower = set(skill.lower().strip() for skill in job_skills)
        candidate_skills_lower = set(skill.lower().strip() for skill in candidate_skills)

        matching_skills = job_skills_lower & candidate_skills_lower
        match_percentage = len(matching_skills) / len(job_skills_lower) * 100

        # Dynamic weight adjustment based on match percentage
        if match_percentage >= 80:
            weight_delta = 0.15    # Exceptional match
        elif match_percentage >= 70:
            weight_delta = 0.10    # Very good match
        elif match_percentage >= 60:
            weight_delta = 0.05    # Good match
        elif match_percentage >= 50:
            weight_delta = 0.02    # Decent match
        elif match_percentage >= 40:
            weight_delta = 0.0     # Average match
        else:
            weight_delta = -0.05   # Poor match penalty

        # Evaluate score alignment with skills
        if model_score is not None:
            if match_percentage >= 70:
                expected_range = (70, 100)
            elif match_percentage >= 40:
                expected_range = (50, 80)
            else:
                expected_range = (0, 60)

            if expected_range[0] <= model_score <= expected_range[1]:
                score = 1   # Good alignment
            elif abs(model_score - np.mean(expected_range)) <= 20:
                score = 0   # Reasonable alignment
            else:
                score = -1  # Poor alignment
        else:
            score = 0  # Cannot evaluate without model score

        return score, weight_delta

    def evaluate_experience_evaluation(self, model_score: Optional[int], model_status: Optional[str],
                                     ground_truth_data: Dict[str, Any]) -> Tuple[int, float]:
        """Evaluate experience assessment with dynamic weight adjustment"""
        # Try to extract experience from ground truth or infer from score
        relevant_years = 0

        # Check if experience info is available in evaluation JSON
        evaluation_json_str = ground_truth_data.get('evaluation_json', '')
        if evaluation_json_str:
            try:
                eval_data = json.loads(evaluation_json_str)
                exp_data = eval_data.get('experience_relevance', {})
                exp_explanation = exp_data.get('explanation', '').lower()

                # Try to extract years from explanation
                year_match = re.search(r'(\d+)\s*years?', exp_explanation)
                if year_match:
                    relevant_years = int(year_match.group(1))
                else:
                    # Infer from experience score
                    exp_score = exp_data.get('score', 0)
                    if exp_score >= 90:
                        relevant_years = 10  # Senior level
                    elif exp_score >= 80:
                        relevant_years = 6   # Mid-level
                    elif exp_score >= 60:
                        relevant_years = 3   # Junior level
                    elif exp_score >= 40:
                        relevant_years = 1   # Entry level
                    else:
                        relevant_years = 0   # No relevant experience
            except (json.JSONDecodeError, TypeError, KeyError):
                # If no experience data, infer from overall score
                if model_score is not None:
                    if model_score >= 85:
                        relevant_years = 8
                    elif model_score >= 70:
                        relevant_years = 5
                    elif model_score >= 50:
                        relevant_years = 2
                    else:
                        relevant_years = 0

        # Calculate weight delta based on experience level
        if relevant_years == 0:
            weight_delta = -0.08
            expected_score_range = (20, 50)
        elif relevant_years == 1:
            weight_delta = -0.05
            expected_score_range = (35, 65)
        elif 2 <= relevant_years <= 3:
            weight_delta = 0.02
            expected_score_range = (50, 75)
        elif 4 <= relevant_years <= 6:
            weight_delta = 0.06
            expected_score_range = (65, 85)
        elif 7 <= relevant_years <= 10:
            weight_delta = 0.10
            expected_score_range = (75, 90)
        else:  # 11+ years
            weight_delta = 0.13
            expected_score_range = (80, 95)

        # Evaluate score alignment
        if model_score is not None:
            score_alignment = expected_score_range[0] <= model_score <= expected_score_range[1]

            # Check status consistency
            status_consistency = True
            if model_status == 'SELECTED' and model_score < 30:
                status_consistency = False
            elif model_status == 'REJECTED' and model_score >= 95:
                status_consistency = False

            if score_alignment and status_consistency:
                score = 1
            elif score_alignment or status_consistency:
                score = 0
            else:
                score = -1
                weight_delta = max(-0.10, weight_delta - 0.08)  # Apply penalty
        else:
            score = 0

        return score, max(-0.10, min(0.18, weight_delta))

    def evaluate_education_assessment(self, model_score: Optional[int]) -> Tuple[int, float]:
        """Evaluate education assessment with dynamic weight adjustment"""
        # Since education info is typically missing from dataset,
        # we'll infer education level from score patterns

        if model_score is None:
            return 0, 0.0

        # Infer education level from score (heuristic approach)
        if model_score >= 85:
            # High score suggests advanced education
            weight_delta = 0.08
            expected_alignment = True
        elif model_score >= 70:
            # Good score suggests bachelor's level
            weight_delta = 0.02
            expected_alignment = True
        elif model_score >= 50:
            # Medium score suggests some formal education
            weight_delta = 0.0
            expected_alignment = True
        else:
            # Lower scores might indicate education mismatch
            weight_delta = -0.02
            expected_alignment = False

        # Since we're inferring, give neutral score unless clear mismatch
        if expected_alignment:
            score = 0  # Neutral (can't verify without actual education data)
        else:
            score = -1 # Penalize only clear mismatches

        return score, weight_delta

    def evaluate_explanation_quality(self, explanation: str, job_skills: List[str]) -> int:
        """Evaluate quality of explanation"""
        if not explanation or len(explanation.strip()) < 10:
            return -1  # Too short or missing

        explanation_lower = explanation.lower()
        components_score = 0

        # Check for skills discussion
        if job_skills:
            skill_mentions = sum(1 for skill in job_skills
                               if skill.lower() in explanation_lower)
            if skill_mentions >= len(job_skills) * 0.3:
                components_score += 1

        # Check for experience discussion
        if any(term in explanation_lower for term in ['experience', 'years', 'background']):
            components_score += 1

        # Check for reasoning
        if any(term in explanation_lower for term in ['because', 'due to', 'based on', 'since']):
            components_score += 1

        if components_score >= 3:
            return 1   # Comprehensive
        elif components_score >= 2:
            return 0   # Adequate
        else:
            return -1  # Poor

    def calculate_single_reward(self, completion: str, ground_truth_data: Dict[str, Any]) -> Dict[str, Any]:
        """Calculate reward for a single completion"""
        # Extract model output
        parsed_output = self.extract_model_output(completion)

        # Extract skills and other data
        job_skills_str = ground_truth_data.get('job_description_skills', '')
        evaluation_json_str = ground_truth_data.get('evaluation_json', '')
        job_skills, candidate_skills = self.extract_skills_from_data(job_skills_str, evaluation_json_str)

        # Get ground truth score
        ground_truth_score = None
        if evaluation_json_str:
            try:
                eval_data = json.loads(evaluation_json_str)
                ground_truth_score = eval_data.get('overall_score')
            except (json.JSONDecodeError, TypeError):
                pass

        # Get expected status
        expected_status = ground_truth_data.get('status', '').replace('STRONGLY_', '').replace('_CONSIDER', '')
        if expected_status not in ['SELECTED', 'REJECTED']:
            # Map status values
            status_mapping = {
                'CONSIDER': 'SELECTED',
                'MAYBE': 'SELECTED',
                'UNLIKELY': 'REJECTED',
                'REJECT': 'REJECTED'
            }
            expected_status = status_mapping.get(expected_status)

        # Evaluate each parameter
        scores = EvaluationScores()

        scores.score_accuracy = self.evaluate_score_accuracy(
            parsed_output['model_score'], ground_truth_score
        )

        scores.status_correctness = self.evaluate_status_correctness(
            parsed_output['model_status'], expected_status, parsed_output['model_score']
        )

        scores.skills_matching, scores.skills_weight_delta = self.evaluate_skills_matching(
            job_skills, candidate_skills, parsed_output['model_score']
        )

        scores.experience_evaluation, scores.experience_weight_delta = self.evaluate_experience_evaluation(
            parsed_output['model_score'], parsed_output['model_status'], ground_truth_data
        )

        scores.education_assessment, scores.education_weight_delta = self.evaluate_education_assessment(
            parsed_output['model_score']
        )

        scores.explanation_quality = self.evaluate_explanation_quality(
            parsed_output['model_explanation'], job_skills
        )

        # Store parsed output
        scores.model_score = parsed_output['model_score']
        scores.model_status = parsed_output['model_status']
        scores.model_explanation = parsed_output['model_explanation']

        # Calculate final reward with dynamic weights
        adjusted_weights = self.weights.copy()
        adjusted_weights['skills_matching'] += scores.skills_weight_delta
        adjusted_weights['experience_evaluation'] += scores.experience_weight_delta
        adjusted_weights['education_assessment'] += scores.education_weight_delta

        # Ensure weights remain positive and normalize
        for key in adjusted_weights:
            adjusted_weights[key] = max(0.01, adjusted_weights[key])
        total_weight = sum(adjusted_weights.values())
        adjusted_weights = {k: v/total_weight for k, v in adjusted_weights.items()}

        # Calculate weighted reward
        parameter_scores = {
            'score_accuracy': scores.score_accuracy,
            'status_correctness': scores.status_correctness,
            'skills_matching': scores.skills_matching,
            'experience_evaluation': scores.experience_evaluation,
            'education_assessment': scores.education_assessment,
            'explanation_quality': scores.explanation_quality
        }

        final_reward = sum(adjusted_weights[param] * score
                          for param, score in parameter_scores.items())

        return {
            'reward': final_reward,
            'scores': scores,
            'parameter_scores': parameter_scores,
            'original_weights': self.weights,
            'adjusted_weights': adjusted_weights,
            'parsed_output': parsed_output
        }


def grpo_compatible_resume_reward_function(
    prompts: List[str],
    completions: List[str],
    completions_ids: Optional[List[List[int]]] = None,
    trainer_state = None,
    **kwargs
) -> List[float]:
    """
    GRPO-compatible reward function for resume evaluation.

    Args:
        prompts: List of input prompts (required by GRPO)
        completions: List of generated completions (required by GRPO)
        completions_ids: List of tokenized completions (optional)
        trainer_state: Current trainer state (optional)
        **kwargs: Dataset columns including evaluation_json, status, etc.

    Returns:
        List of reward scores (one per completion)
    """
    # Initialize reward function
    reward_function = GRPOResumeRewardFunction()

    # Extract dataset columns from kwargs
    evaluation_jsons = kwargs.get('evaluation_json', [None] * len(completions))
    statuses = kwargs.get('status', [None] * len(completions))
    job_description_skills = kwargs.get('job_description_skills', [None] * len(completions))
    similarity_scores = kwargs.get('similarity_score', [0.0] * len(completions))
    confidence_scores = kwargs.get('confidence_score', [0.0] * len(completions))

    # Calculate rewards
    rewards = []

    for i, completion in enumerate(completions):
        try:
            # Prepare ground truth data for this sample
            ground_truth_data = {
                'evaluation_json': evaluation_jsons[i] if i < len(evaluation_jsons) else None,
                'status': statuses[i] if i < len(statuses) else None,
                'job_description_skills': job_description_skills[i] if i < len(job_description_skills) else None,
                'similarity_score': similarity_scores[i] if i < len(similarity_scores) else 0.0,
                'confidence_score': confidence_scores[i] if i < len(confidence_scores) else 0.0,
            }

            # Calculate reward for this completion
            result = reward_function.calculate_single_reward(completion, ground_truth_data)
            reward = result['reward']

            # Apply training progress scaling if trainer_state is available
            if trainer_state is not None:
                progress = trainer_state.global_step / max(trainer_state.max_steps, 1000)
                progress_factor = 0.5 + 0.5 * progress  # Scale from 0.5 to 1.0
                reward *= progress_factor

            rewards.append(float(reward))

        except Exception as e:
            print(f"Warning: Error calculating reward for completion {i}: {e}")
            rewards.append(0.0)  # Neutral reward for errors

    return rewards

In [5]:
import json
from copy import deepcopy

# ---------- Test Sample with New Education Fields ----------
import json
from copy import deepcopy

# ---------- One complete ground-truth record ----------
test_record = {
    "status": "REJECT",                          # ground-truth hiring status
    "job_description_skills": "Python, SQL, ML", # comma-separated job skills
    "evaluation_json": json.dumps({
        "overall_score": 82,
        "skills_match": {
            "present_skills": ["Python", "ML"],
            "missing_skills": ["SQL"]
        },
        "experience_relevance": {
            "score": 75,
            "explanation": "4 years of machine-learning engineering experience."
        },
        # ✅ Newly added nested education evaluation
        "education_evaluation": {
            "score": 90,
            "explanation": "Master’s in Data Science meets job requirements."
        }
    }),
    # ✅ Newly added top-level education block
    "education": {
        "highest_degree": "Master's",
        "major": "Data Science",
        "institution": "IIT Bombay",
        "graduation_year": 2022,
        "gpa": 8.7
    },
    "similarity_score": 0.41,
    "confidence_score": 0.73,
}


# ---------- Simulated model completion ----------
completion_text = """
The evaluation:
{
  "overall_score": 75,
  "status": "REJECTED",
  "explanation": "Candidate strong in frontend but lacks Node.js experience."
}
"""

# ---------- Run the reward calculation ----------
reward_fn = GRPOResumeRewardFunction()

result = reward_fn.calculate_single_reward(
    completion=completion_text,
    ground_truth_data=deepcopy(test_record)
)

print("Parsed Output:\n", json.dumps(result["parsed_output"], indent=2))
print("\nParameter Scores:\n", json.dumps(result["parameter_scores"], indent=2))
print("\nFinal Reward:", result["reward"])
print("\nEducation top-level:\n", test_record.get("education"))


Parsed Output:
 {
  "model_score": 75,
  "model_status": "REJECTED",
  "model_explanation": "Candidate strong in frontend but lacks Node.js experience.",
  "raw_json": {
    "overall_score": 75,
    "status": "REJECTED",
    "explanation": "Candidate strong in frontend but lacks Node.js experience."
  }
}

Parameter Scores:
 {
  "score_accuracy": 1,
  "status_correctness": 1,
  "skills_matching": 1,
  "experience_evaluation": 1,
  "education_assessment": 0,
  "explanation_quality": -1
}

Final Reward: 0.7168141592920353

Education top-level:
 {'highest_degree': "Master's", 'major': 'Data Science', 'institution': 'IIT Bombay', 'graduation_year': 2022, 'gpa': 8.7}


In [ ]:
# Example usage and testing
def test_grpo_reward_function():
    """Test the GRPO-compatible reward function"""

    # Example data matching your dataset format
    prompts = [
        "Evaluate this candidate profile against the job requirements and provide a score (0-100), status (SELECTED/REJECTED), and explanation."
    ]

    completions = [
        '{"score": 84, "status": "SELECTED", "explanation": "Strong technical skills with good automation and SQL experience. Vue.js and Docker knowledge align well with requirements. Solid overall fit for the role."}'
    ]

    # Dataset columns (kwargs)
    kwargs = {
        'evaluation_json': ['{"overall_score": 84, "skills_match": {"score": 96, "present_skills": ["automation", "sql", "selenium", "testing", "vue.js", "adaptability", "jira", "docker"], "missing_skills": []}, "experience_relevance": {"score": 79, "explanation": "Solid experience level, suitable for the role."}}'],
        'status': ['STRONGLY_CONSIDER'],
        'job_description_skills': ['SQL, Vue.js, Selenium, Docker, Jira, Adaptability, Automation, Testing'],
        'similarity_score': [0.84],
        'confidence_score': [0.87]
    }

    # Test the reward function
    rewards = grpo_compatible_resume_reward_function(
        prompts=prompts,
        completions=completions,
        **kwargs
    )

    print("=== GRPO Resume Reward Function Test ===")
    print(f"Input completion: {completions[0]}")
    print(f"Calculated reward: {rewards[0]:.4f}")

    # Test with detailed analysis
    reward_function = GRPOResumeRewardFunction()
    ground_truth = {
        'evaluation_json': kwargs['evaluation_json'][0],
        'status': kwargs['status'][0],
        'job_description_skills': kwargs['job_description_skills'][0],
        'similarity_score': kwargs['similarity_score'][0],
        'confidence_score': kwargs['confidence_score'][0]
    }

    detailed_result = reward_function.calculate_single_reward(completions[0], ground_truth)

    print("\n=== Detailed Analysis ===")
    print(f"Final Reward: {detailed_result['reward']:.4f}")
    print("\nParameter Scores:")
    for param, score in detailed_result['parameter_scores'].items():
        orig_weight = detailed_result['original_weights'][param]
        adj_weight = detailed_result['adjusted_weights'][param]
        contribution = adj_weight * score
        print(f"  {param}: {score} (weight: {orig_weight:.2f} → {adj_weight:.2f}, contribution: {contribution:.3f})")

    print(f"\nModel Output:")
    parsed = detailed_result['parsed_output']
    print(f"  Score: {parsed['model_score']}")
    print(f"  Status: {parsed['model_status']}")
    print(f"  Explanation: {parsed['model_explanation']}")

    return rewards


if __name__ == "__main__":
    test_grpo_reward_function()

=== GRPO Resume Reward Function Test ===
Input completion: {"score": 84, "status": "SELECTED", "explanation": "Strong technical skills with good automation and SQL experience. Vue.js and Docker knowledge align well with requirements. Solid overall fit for the role."}
Calculated reward: 0.6723

=== Detailed Analysis ===
Final Reward: 0.6723

Parameter Scores:
  score_accuracy: 1 (weight: 0.25 → 0.21, contribution: 0.210)
  status_correctness: 1 (weight: 0.20 → 0.17, contribution: 0.168)
  skills_matching: 1 (weight: 0.20 → 0.29, contribution: 0.294)
  experience_evaluation: 0 (weight: 0.15 → 0.14, contribution: 0.000)
  education_assessment: 0 (weight: 0.10 → 0.10, contribution: 0.000)
  explanation_quality: 0 (weight: 0.10 → 0.08, contribution: 0.000)

Model Output:
  Score: 84
  Status: SELECTED
  Explanation: Strong technical skills with good automation and SQL experience. Vue.js and Docker knowledge align well with requirements. Solid overall fit for the role.


# GRPO Pipeline

In [ ]:
import wandb

wandb.login()

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: shreyanshjaino6ai (o6ailabs) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
import json
import wandb
from datasets import Dataset
from trl import GRPOTrainer, GRPOConfig

# ================================
# 1. PROMPT GENERATION FUNCTION
# ================================
def create_prompt(job_skills):
    prompt = f"""You are an HR expert evaluating candidate resumes.

Job Requirements: {job_skills}

Evaluate the candidate and provide your assessment in JSON format:
{{"score": [0-100], "status": "SELECTED" or "REJECTED", "explanation": "your reasoning"}}

Your evaluation:"""
    return prompt

# ================================
# 2. DATA LOADING & PREPARATION
# ================================
def load_and_prepare_dataset(data_file_path):
    print(f"Loading data from {data_file_path}...")

    with open(data_file_path, 'r') as f:
        if data_file_path.endswith('.jsonl'):
            raw_data = [json.loads(line.strip()) for line in f if line.strip()]
        else:
            raw_data = json.load(f)
            if isinstance(raw_data, dict):
                raw_data = [raw_data]

    print(f"Loaded {len(raw_data)} samples")

    dataset_samples = []

    for item in raw_data:
        job_skills = item.get('job_description_skills', '')
        prompt = create_prompt(job_skills)

        sample = {
            'prompt': prompt,
            'evaluation_json': item.get('evaluation_json', ''),
            'status': item.get('status', ''),
            'job_description_skills': job_skills,
            'similarity_score': item.get('similarity_score', 0.0),
            'confidence_score': item.get('confidence_score', 0.0),
            'candidate_name': item.get('candidate_name', ''),
            'id': item.get('id', ''),
        }

        dataset_samples.append(sample)

    dataset = Dataset.from_list(dataset_samples)

    print("✅ Dataset prepared!")
    print(f"Columns: {dataset.column_names}")
    print(f"Sample prompt preview: {dataset[0]['prompt'][:150]}...")

    return dataset

# ================================
# 3. GRPO CONFIGURATION
# ================================
def setup_grpo_config():
    return GRPOConfig(
        learning_rate=1e-5,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        beta=0.1,
        num_train_epochs=2,
        # eval_steps=50,
        save_steps=100,
        logging_steps=5,
        output_dir="./grpo_resume_model",
        logging_dir="./logs",
        report_to="wandb",
        gradient_checkpointing=True,
        dataloader_pin_memory=False,
        remove_unused_columns=False,
        save_strategy="steps",
    )

# ================================
# 4. MAIN GRPO TRAINING FUNCTION
# ================================
def run_grpo_training(train_data_path, eval_data_path=None, model=None, tokenizer=None):
    print("🚀 Starting GRPO Training Setup...")

    if model is None or tokenizer is None:
        raise ValueError("❌ Please provide a pre-loaded model and tokenizer.")

    # Prepare datasets
    train_dataset = load_and_prepare_dataset(train_data_path)

    eval_dataset = None
    if eval_data_path:
        eval_dataset = load_and_prepare_dataset(eval_data_path)

    training_args = setup_grpo_config()

    print("Setting up GRPO trainer...")

    trainer = GRPOTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        # eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        reward_funcs=grpo_compatible_resume_reward_function,
    )

    wandb.init(
        project="o6ai-resume-evaluation-GRPO",
        name="grpo-qwen-resume-eval",
        config={
            "base_model": "Qwen2-0.5B",
            "training_type": "GRPO",
            "dataset_size": len(train_dataset),
            "reward_function": "6-parameter-evaluation"
        }
    )

    print("Starting training...")
    trainer.train()

    print("Saving trained model...")
    trainer.save_model("./final_grpo_resume_model")

    print("Training completed successfully!")
    return trainer



# Run GRPO training

In [ ]:
run_grpo_training(
    train_data_path="/content/o6ai-agent-hr/synthetic_resume_eval_data_1000.json",
    # eval_data_path="/content/eval_data.json",  # optional
    model=model,
    tokenizer=tokenizer
)

🚀 Starting GRPO Training Setup...
Loading data from /content/o6ai-agent-hr/synthetic_resume_eval_data_1000.json...
Loaded 1000 samples
✅ Dataset prepared!
Columns: ['prompt', 'evaluation_json', 'status', 'job_description_skills', 'similarity_score', 'confidence_score', 'candidate_name', 'id']
Sample prompt preview: You are an HR expert evaluating candidate resumes.

Job Requirements: JavaScript, Java, Scrum, Adaptability, Microservices, Python, REST APIs

Evaluat...
Unsloth: We now expect `per_device_train_batch_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 8
Setting up GRPO trainer...


🎯 Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 2 | Total steps = 250
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 8 x 1) = 64
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,entropy,rewards / grpo_compatible_resume_reward_function / mean,rewards / grpo_compatible_resume_reward_function / std
5,0.114800,-0.155168,0.093351,184.243750,23.400000,256.000000,0.481250,117.377280,23.400000,240.600000,1.148166,0,-0.155168,0.190558
10,0.115600,-0.088041,0.083007,195.359375,22.400000,256.000000,0.562500,117.550844,22.400000,244.600000,1.156072,No Log,-0.088041,0.192256
15,0.125000,-0.143363,0.085166,173.168750,24.000000,256.000000,0.437500,109.331955,24.000000,244.000000,1.250454,No Log,-0.143363,0.198165
20,0.090500,-0.138683,0.096011,151.143750,17.000000,256.000000,0.356250,93.995573,17.000000,241.200000,0.905297,No Log,-0.138683,0.189234
25,0.096700,-0.095675,0.127206,116.237500,23.000000,256.000000,0.221875,76.517130,23.000000,242.000000,0.967494,No Log,-0.095675,0.209572
30,0.186400,-0.133797,0.140478,86.425000,20.200000,256.000000,0.109375,65.476765,20.200000,206.400000,1.863679,No Log,-0.133797,0.221037
35,0.100900,-0.065704,0.161988,104.809375,28.600000,256.000000,0.178125,72.330687,28.600000,201.000000,1.009467,No Log,-0.065704,0.237727
40,0.093100,-0.075182,0.174224,118.559375,24.800000,256.000000,0.218750,79.918211,24.800000,232.200000,0.931236,No Log,-0.075182,0.209302
45,0.139300,-0.034824,0.188824,114.293750,18.400000,256.000000,0.146875,89.943466,18.400000,235.000000,1.392671,No Log,-0.034824,0.220060
50,0.065700,-0.013440,0.185579,112.150000,26.400000,256.000000,0.153125,86.172278,26.400000,235.600000,0.657018,No Log,-0.013440,0.202888


💾 Saving trained model...
✅ Training completed successfully!
